In [1]:
import polars as pl
import os
import glob
from tqdm import tqdm

## Functions

In [2]:
def reorder_cols(df: pl.DataFrame) -> pl.DataFrame:
	"""
	Reorders the columns of a DataFrame so that 'sample' is the first column if it exists.
	"""
	cols = df.columns

	if "sample" in cols:
		cols.remove("sample")
		cols.insert(0, "sample")

	return df[cols]

In [3]:
def summarize_res(df: pl.DataFrame) -> pl.DataFrame:
	
	bool_cols = df['filter_1_mutation_intra_hairpin_loop':'filter_8_low_quality'].columns
	str_cols = df['msec_filter_123':'msec_filter_all'].columns
	n_variants = df.height

	# Initialize a dictionary
	results = {
		"filter": [],
		"percentage": [],
		"n_failed": []
	}

	# Populate the dictionary inside the loops
	for col in bool_cols:
		n_failed = df[col].sum() # Sum counts True values
		
		results["filter"].append(col)
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	for col in str_cols:
		n_failed = df.filter(~pl.col(col).is_null()).height # Count non-nulls
		
		results["filter"].append(col)
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	# Create DataFrame from the dictionary
	return pl.DataFrame(results).with_columns(pl.lit(n_variants).alias("total_variants"))

### MicroSEC filter Description

The MicroSEC pipeline contains 8 filtering processes.  

- Filter 1  : Shorter-supporting lengths distribute too short to occur (1-1 and 1-2).  
	- Filter 1-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 1-2: The shorter-supporting lengths distributed over less than 75% of the read length.  
- Filter 2  : Hairpin-structure induced error detection (2-1 and 2-2).  
	- Filter 2-1: Palindromic sequences exist within 200 bases.  
	- Filter 2-2: >=50% mutation-supporting reads contains a reverse complementary sequence of the opposite strand consisting >= 15 bases.  
- Filter 3  : 3'-/5'-supporting lengths are too densely distributed to occur (3-1 and 3-2).  
	- Filter 3-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 3-2: The distributions of 3'-/5'-supporting lengths are within 75% of the read length.  
- Filter 4  : >=15% mutations were called by chimeric reads comprising two distant regions.  
- Filter 5  : >=50% mutations were called by soft-clipped reads.  
- Filter 6  : Mutations locating at simple repeat sequences.  
- Filter 7  : Indel mutations locating at a >=15 homopolymer.  
- Filter 8  : >=10% of bases are low quality (Quality score <18) in the mutation supporting reads.  

Filter 1, 2, 3, and 4 detect possible FFPE artifacts.  
Filter 5 may also be FFPE artifacts or mapping errors.  
Filter 6, 7, and 8 detect frequent errors caused by the next generation sequencing platform.  
Supporting lengths are adjusted considering small repeat sequences around the mutations.  
  
Results are saved in a tsv file.  

github url: https://github.com/MANO-B/MicroSEC

## Main
### VCF

In [23]:
msec_paths = sorted(glob.glob("../vcf-micr-svf/*/*.microsec.tsv"))

all_res = []

for path in tqdm(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t", infer_schema_length = 1000).rename(lambda x : x.lower())
	
	all_res.append(df)
	
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

100%|██████████| 1641/1641 [03:08<00:00,  8.73it/s]


In [24]:
final_df

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1001159-01""","""12-del""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1001159-01""","""1-snv""","""chr1""",16262742,"""G""","""A""","""N""","""CTCTGTTGGCCTGCCTTCCCAGACCAAGAC…",49,772,2,0,48,48,24,524,399,0.012821,0.015803,0.017098,0.0,0.002591,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""1-snv""","""chr1""",45797760,"""T""","""C""","""N""","""AGAGCTGTTCCTGCTCCACCCGAGAGGCAC…",49,928,4,0,48,48,24,490,415,0.015306,0.019935,0.014763,0.0,0.00431,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""1-snv""","""chr10""",43601830,"""G""","""A""","""N""","""TCTGCATCCTGCAGGACACCATGGTGGCCA…",49,451,1,0,48,48,24,355,380,0.011675,0.013969,0.013969,0.0,0.002217,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""1-snv""","""chr11""",118307334,"""C""","""T""","""N""","""CGCCCCGCGGCAACGCGTCCTGGCCCTGCT…",49,34,0,0,46,45,23,174,88,0.006603,0.008824,0.008824,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""1-snv""","""chr17""",37856504,"""G""","""A""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""1-snv""","""chr22""",29108003,"""C""","""T""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""1-snv""","""chr4""",1920021,"""A""","""C""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null


In [25]:
## Samples that could be processed
final_df["sample"].unique()

sample
str
"""ORD-1691406-01"""
"""ORD-1050682-01"""
"""ORD-1959107-01"""
"""ORD-1789900-01"""
"""ORD-1748047-01"""
…
"""ORD-1526119-01"""
"""ORD-1800292-01"""
"""ORD-1742473-01"""


In [26]:
# All Filters
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1001159-01""","""12-del""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1001159-01""","""12-del""","""chr3""",41266087,"""GTCTTACCTGGAC""","""G""","""N""","""GTTAGTCACTGGCAGCAACAGTCTGGAATC…",49,117,2,0,28,44,28,341,302,0.009942,0.011111,0.011966,0.0,0.017094,1.9328e-18,1.7498e-34,1.0025e-32,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-1020342-01""","""12-del""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",144,31785,5347,0,121,139,78,716,667,0.011361,0.012632,0.012122,0.0,0.168224,0.0,0.0,0.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",""" filter 1: p is small, but sup…"
"""ORD-1042000-01""","""1-del""","""chr11""",118359946,"""CA""","""C""","""N""","""GACACAGTGAGACTCCATCTCAAAAAAAAA…",49,29,1,0,20,48,20,20,243,0.107671,0.0,0.010345,0.068966,0.034483,0.101477,0.000964,0.026523,false,false,false,false,false,false,true,true,null,null,"""Artifact suspicious""",null
"""ORD-1042000-01""","""1-ins""","""chr12""",18691260,"""G""","""GA""","""N""","""GGAAATGGTAAGTCCCTTGGGAAAAAAAAA…",49,106,0,0,33,41,24,250,254,0.053138,0.062264,0.067925,0.0,0.0,1.8268e-18,3.8519e-14,2.4730e-13,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 1: p is small, but sup…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""1-snv""","""chr17""",29483000,"""G""","""T""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-2131606-01""","""2-ins""","""chr8""",117864264,"""C""","""CAG""","""N""","""TGGTGGAGGCATAGCTGACTCAGATCTATG…",144,37,26,0,118,112,56,145,112,0.010323,0.005405,0.010811,0.0,0.702703,9.2195e-9,0.000591,0.001694,true,false,false,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null


1184

In [27]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1001159-01""","""12-del""","""chr3""",41266087,"""GTCTTACCTGGAC""","""G""","""N""","""GTTAGTCACTGGCAGCAACAGTCTGGAATC…",49,117,2,0,28,44,28,341,302,0.009942,0.011111,0.011966,0.0,0.017094,1.9328e-18,1.7498e-34,1.0025e-32,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-1042000-01""","""1-ins""","""chr12""",18691260,"""G""","""GA""","""N""","""GGAAATGGTAAGTCCCTTGGGAAAAAAAAA…",49,106,0,0,33,41,24,250,254,0.053138,0.062264,0.067925,0.0,0.0,1.8268e-18,3.8519e-14,2.4730e-13,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 1: p is small, but sup…"
"""ORD-1042000-01""","""1-ins""","""chr17""",41204899,"""T""","""TG""","""N""","""TAGGACAAGTCTGTGTGTTTTGTTTTTTTT…",49,31,0,0,34,39,24,231,165,0.14944,0.03871,0.232258,0.0,0.0,1.6373e-7,2.8218e-11,7.3711e-11,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but rea…"
"""ORD-1048486-01""","""1-snv""","""chr4""",55140704,"""G""","""A""","""N""","""CTCTTGTCACGTAGCCCTGCATTCTGAACT…",144,54,34,0,68,115,68,170,345,0.00643,0.005556,0.007407,0.0,0.62963,0.000311,1.0089e-9,1.5114e-18,false,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1049250-01""","""1-ins""","""chr2""",61726060,"""A""","""AC""","""N""","""TTGCACATGCTAAAAAAAAAACACACACAA…",49,215,1,0,43,38,24,396,541,0.026578,0.030233,0.027442,0.0,0.004651,6.7928e-16,2.6807e-34,4.2662e-32,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 1: p is small, but sup…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""1-snv""","""chr3""",178952085,"""A""","""G""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""1-snv""","""chr2""",25463568,"""A""","""G""","""N""","""CATTGCAGGGACTGCCCCCAGTCACCAGAT…",144,56,2,0,95,117,71,156,117,0.008433,0.010714,0.0125,0.0,0.035714,3.0944e-7,4.7255e-11,9.2555e-11,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 1: p is small, but sup…"
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"


652

In [31]:
summarize_res(final_df)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",0.914739,424,46352
"""filter_2_hairpin_structure""",0.030204,14,46352
"""filter_3_microhomology_induced…",1.382896,641,46352
"""filter_4_highly_homologous_reg…",1.279341,593,46352
"""filter_5_soft_clipped_reads""",0.509147,236,46352
…,…,…,…
"""filter_7_mutation_at_homopolym…",2.524163,1170,46352
"""filter_8_low_quality""",3.21669,1491,46352
"""msec_filter_123""",1.876942,870,46352


In [32]:
summarize_res(arti_filter_1234)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",29.302004,424,1447
"""filter_2_hairpin_structure""",0.967519,14,1447
"""filter_3_microhomology_induced…",44.298549,641,1447
"""filter_4_highly_homologous_reg…",40.981341,593,1447
"""filter_5_soft_clipped_reads""",4.215619,61,1447
…,…,…,…
"""filter_7_mutation_at_homopolym…",11.817554,171,1447
"""filter_8_low_quality""",14.098134,204,1447
"""msec_filter_123""",60.124395,870,1447


In [30]:
summarize_res(arti_all_filter)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",8.767577,424,4836
"""filter_2_hairpin_structure""",0.289495,14,4836
"""filter_3_microhomology_induced…",13.254756,641,4836
"""filter_4_highly_homologous_reg…",12.2622,593,4836
"""filter_5_soft_clipped_reads""",4.880066,236,4836
…,…,…,…
"""filter_7_mutation_at_homopolym…",24.193548,1170,4836
"""filter_8_low_quality""",30.831266,1491,4836
"""msec_filter_123""",17.990074,870,4836


### XML

In [34]:
msec_paths = sorted(glob.glob("../xml-micr-svf/*/*.microsec.tsv"))
len(msec_paths)

1606

In [35]:
all_res = []

for path in tqdm(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t").rename(lambda x: x.lower())
	
	all_res.append(df)
	
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

100%|██████████| 1606/1606 [00:50<00:00, 31.66it/s]


In [36]:
final_df

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1001159-01""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""MTOR""",true,1053,"""5490_5501delTGCCGCCACCAC""","""T1834_T1837del""",0.3922,"""nonframeshift""","""NM_004958""","""-""",false,"""12-del""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1001159-01""","""chr1""",16262742,"""G""","""A""","""SPEN""",true,902,"""10007G>A""","""R3336Q""",0.5488,"""missense""","""NM_015001""","""+""",false,"""1-snv""","""N""","""CTCTGTTGGCCTGCCTTCCCAGACCAAGAC…",49,772,2,0,48,48,24,524,399,0.012821,0.015803,0.017098,0.0,0.002591,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""chr1""",45797760,"""T""","""C""","""MUTYH""",false,1044,"""892-2A>G""","""splice site 892-2A>G""",0.5354,"""splice""","""NM_001048171""","""-""",false,"""1-snv""","""N""","""AGAGCTGTTCCTGCTCCACCCGAGAGGCAC…",49,928,4,0,48,48,24,490,415,0.015306,0.019935,0.014763,0.0,0.00431,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""chr10""",43601830,"""G""","""A""","""RET""",true,631,"""874G>A""","""V292M""",0.4517,"""missense""","""NM_020975""","""+""",false,"""1-snv""","""N""","""TCTGCATCCTGCAGGACACCATGGTGGCCA…",49,451,1,0,48,48,24,355,380,0.011675,0.013969,0.013969,0.0,0.002217,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""chr11""",118307334,"""C""","""T""","""MLL""",true,31,"""107C>T""","""P36L""",0.9032,"""missense""","""NM_005933""","""+""",false,"""1-snv""","""N""","""CGCCCCGCGGCAACGCGTCCTGGCCCTGCT…",49,34,0,0,46,45,23,174,88,0.006603,0.008824,0.008824,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""chr17""",37856504,"""G""","""A""","""ERBB2""",true,3682,"""13G>A""","""A5T""",0.0019,"""missense""","""NM_004448""","""+""",false,"""1-snv""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""chr22""",29108003,"""C""","""T""","""CHEK2""",true,3729,"""686G>A""","""G229D""",0.0013,"""missense""","""NM_007194""","""-""",false,"""1-snv""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""chr4""",1920021,"""A""","""C""","""WHSC1""",true,1979,"""1081A>C""","""K361Q""",0.5073,"""missense""","""NM_133335""","""+""",false,"""1-snv""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,fal

In [16]:
## Samples that could be processed
final_df["sample"].unique()

sample
str
"""ORD-1993016-01"""
"""ORD-2104088-01"""
"""ORD-2069193-01"""
"""ORD-1815132-01"""
"""ORD-1562107-01"""
…
"""ORD-1296818-01"""
"""ORD-1867608-01"""
"""ORD-1764534-01"""


In [17]:
# All artifacts
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1001159-01""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""MTOR""",true,1053,"""5490_5501delTGCCGCCACCAC""","""T1834_T1837del""",0.3922,"""nonframeshift""","""NM_004958""","""-""",false,"""12-del""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1020342-01""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""MTOR""",true,6027,"""5490_5501delTGCCGCCACCAC""","""T1834_T1837del""",0.4007,"""nonframeshift""","""NM_004958""","""-""",false,"""12-del""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",144,31785,5347,0,121,139,78,716,667,0.011361,0.012632,0.012122,0.0,0.168224,0.0,0.0,0.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",""" filter 1: p is small, but sup…"
"""ORD-1048486-01""","""chr4""",55140704,"""G""","""A""","""PDGFRA""",true,3073,"""1565G>A""","""R522H""",0.0026,"""missense""","""NM_006206""","""+""",false,"""1-snv""","""N""","""CTCTTGTCACGTAGCCCTGCATTCTGAACT…",144,54,34,0,68,115,68,170,345,0.00643,0.005556,0.007407,0.0,0.62963,0.000311,1.0089e-9,1.5114e-18,false,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1049250-01""","""chr6""",33287880,"""GCCT""","""G""","""DAXX""",true,654,"""1370_1372delAGG""","""E457del""",0.344,"""nonframeshift""","""NM_001350""","""-""",false,"""3-del""","""Y""","""CCTCCTCTTCAGAATCTGTGGCCTCCTCTT…",49,94,1,0,17,48,17,334,428,0.012809,0.006383,0.006383,0.0,0.010638,1.6307e-21,9.2067e-45,1.0476e-47,true,false,true,false,false,true,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1050682-01""","""chr19""",11098593,"""G""","""T""","""SMARCA4""",false,960,"""1111G>T""","""E371*""",0.0063,"""nonsense""","""NM_003072""","""+""",false,"""1-snv""","""N""","""TGGAGATCCTGCAGGAGCGCTAGTACAGGT…",144,27,16,0,129,90,59,293,295,0.006944,0.018519,0.007407,0.0,0.592593,0.000011,0.000003,0.000002,false,false,false,false,true,false,false,false,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""chr17""",29483000,"""G""","""T""","""NF1""",false,2006,"""61-1G>T""","""splice site 61-1G>T""",0.0957,"""splice""","""NM_001042492""","""+""",false,"""1-snv""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""CUL3""",false,1027,"""67-1_81delGATGACCATGGATGAA""","""splice site 67-1_81delGATGACCA…",0.0351,"""splice""","""NM_003590""","""-""",false,"""16-del""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001

764

In [18]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1048486-01""","""chr4""",55140704,"""G""","""A""","""PDGFRA""",true,3073,"""1565G>A""","""R522H""",0.0026,"""missense""","""NM_006206""","""+""",false,"""1-snv""","""N""","""CTCTTGTCACGTAGCCCTGCATTCTGAACT…",144,54,34,0,68,115,68,170,345,0.00643,0.005556,0.007407,0.0,0.62963,0.000311,1.0089e-9,1.5114e-18,false,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1049250-01""","""chr6""",33287880,"""GCCT""","""G""","""DAXX""",true,654,"""1370_1372delAGG""","""E457del""",0.344,"""nonframeshift""","""NM_001350""","""-""",false,"""3-del""","""Y""","""CCTCCTCTTCAGAATCTGTGGCCTCCTCTT…",49,94,1,0,17,48,17,334,428,0.012809,0.006383,0.006383,0.0,0.010638,1.6307e-21,9.2067e-45,1.0476e-47,true,false,true,false,false,true,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1111445-01""","""chr16""",89880936,"""GA""","""G""","""FANCA""",false,787,"""274delT""","""S92fs*3""",0.0089,"""frameshift""","""NM_000135""","""-""",false,"""1-del""","""N""","""GCTATAACTTACCTATAAATGACTAGAATG…",144,46,6,0,131,71,71,140,71,0.012379,0.01087,0.013043,0.0,0.130435,0.000054,1.0927e-17,9.4506e-18,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1114063-01""","""chr2""",212288966,"""C""","""T""","""ERBB4""",true,1229,"""2780G>A""","""R927Q""",0.0033,"""missense""","""NM_005235""","""-""",false,"""1-snv""","""N""","""CTAATAAATCAGGGATTTCTTGCGTTGGAA…",144,38,14,0,90,133,71,242,133,0.027047,0.034211,0.042105,0.0,0.368421,0.066392,1.7303e-8,3.0432e-8,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1127305-01""","""chr7""",116412029,"""CTACTTTTCCAGAAG""","""C""","""MET""",false,447,"""3015_3028delTACTTTTCCAGAAG""","""T1006fs*4""",0.0895,"""frameshift""","""NM_000245""","""+""",false,"""14-del""","""N""","""TGAATCTGTAGACTACCGAGCGTATATTTC…",49,44,0,0,29,44,29,169,328,0.003711,0.002273,0.0,0.0,0.0,2.3566e-14,1.5735e-14,6.0246e-13,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""chr3""",178952085,"""A""","""G""","""PIK3CA""",false,703,"""3140A>G""","""H1047R""",0.01,"""missense""","""NM_006218""","""+""",false,"""1-snv""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""chr2""",25463568,"""A""","""G""","""DNMT3A""",true,2024,"""2114T>C""","""I705T""",0.004,"""missense""","""NM_022552""","""-""",false,"""1-snv""","""N""","""CATTGCAGGGACTGCCCCCAGTCACC

243

In [19]:
summarize_res(final_df)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",0.763924,165,21599
"""filter_2_hairpin_structure""",0.0,0,21599
"""filter_3_microhomology_induced…",0.648178,140,21599
"""filter_4_highly_homologous_reg…",0.208343,45,21599
"""filter_5_soft_clipped_reads""",0.717626,155,21599
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.00926,2,21599
"""filter_8_low_quality""",1.824159,394,21599
"""msec_filter_123""",1.101903,238,21599


In [20]:
summarize_res(arti_filter_1234)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",58.303887,165,283
"""filter_2_hairpin_structure""",0.0,0,283
"""filter_3_microhomology_induced…",49.469965,140,283
"""filter_4_highly_homologous_reg…",15.90106,45,283
"""filter_5_soft_clipped_reads""",8.480565,24,283
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.0,0,283
"""filter_8_low_quality""",2.826855,8,283
"""msec_filter_123""",84.09894,238,283


In [21]:
summarize_res(arti_all_filter)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",13.924051,165,1185
"""filter_2_hairpin_structure""",0.0,0,1185
"""filter_3_microhomology_induced…",11.814346,140,1185
"""filter_4_highly_homologous_reg…",3.797468,45,1185
"""filter_5_soft_clipped_reads""",13.080169,155,1185
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.168776,2,1185
"""filter_8_low_quality""",33.248945,394,1185
"""msec_filter_123""",20.084388,238,1185
